In [4]:
import random

m, n = map(int, input("Nhap so dong va cot: ").split())
room = []

real_x = random.randint(0, m - 1)
real_y = random.randint(0, n - 1)

for i in range(m):
    row = [random.randint(0, 1) for _ in range(n)]
    room.append(row)

room[real_x][real_y] = 0

print(f"\nVi tri thuc te ban dau của Robot: ({real_x}, {real_y})")

def is_clean(room_state):
    """Kiểm tra xem toàn bộ phòng đã sạch chưa."""
    for row in room_state:
        if 1 in row:
            return False
    return True

def get_percept(x, y):
    """
    CẢM BIẾN: Trả về trạng thái va chạm tường ở 4 hướng (UP, DOWN, LEFT, RIGHT)
    'W' = Wall, 'O' = Open
    Ví dụ: ('W', 'O', 'W', 'O') nghĩa là phía trên và bên trái là tường.
    """
    u = "W" if x == 0 else "O"
    d = "W" if x == m - 1 else "O"
    l = "W" if y == 0 else "O"
    r = "W" if y == n - 1 else "O"
    return (u, d, l, r)

def predict_belief(belief_positions, action):
    next_positions = set()
    for (x, y) in belief_positions:
        if action == "UP":     nx, ny = max(0, x - 1), y
        elif action == "DOWN": nx, ny = min(m - 1, x + 1), y
        elif action == "LEFT": nx, ny = x, max(0, y - 1)
        elif action == "RIGHT":nx, ny = x, min(n - 1, y + 1)
        next_positions.add((nx, ny))
    return frozenset(next_positions)

def update_room_and_beliefs(room_tuple, pred_positions, percept):
    filtered_positions = frozenset((x, y) for (x, y) in pred_positions if get_percept(x, y) == percept)

    next_room = [list(row) for row in room_tuple]
    if len(filtered_positions) == 1:
        ox, oy = list(filtered_positions)[0]
        next_room[ox][oy] = 0

    return tuple(tuple(row) for row in next_room), filtered_positions

def and_or_search(start_room):
    start_room_tuple = tuple(tuple(row) for row in start_room)
    initial_positions = frozenset((i, j) for i in range(m) for j in range(n))

    reached_states = set()

    def or_search(curr_room, curr_beliefs, path):
        state_signature = (curr_room, curr_beliefs)
        reached_states.add(state_signature)

        if is_clean(curr_room):
            return "GOAL"

        if state_signature in path:
            return None

        for action in ["UP", "DOWN", "LEFT", "RIGHT"]:
            pred_beliefs = predict_belief(curr_beliefs, action)

            possible_percepts = set(get_percept(x, y) for (x, y) in pred_beliefs)

            percept_branches = {}
            action_is_valid = True

            for percept in possible_percepts:
                next_room, next_beliefs = update_room_and_beliefs(curr_room, pred_beliefs, percept)

                result = or_search(next_room, next_beliefs, path + [state_signature])

                if result is None:
                    action_is_valid = False
                    break
                percept_branches[percept] = result

            if action_is_valid:
                return (action, percept_branches)

        return None

    plan = or_search(start_room_tuple, initial_positions, [])
    return plan, len(reached_states)

print("\n--- Tìm kiếm với quan sát một phần ---")
plan, total_states = and_or_search(room)

if plan is None:
    print("Không tìm thấy kế hoạch nào để làm sạch phòng trong mọi trường hợp!")
else:
    print(f"Thành công! Đã duyệt {total_states} trạng thái niềm tin.")
    print("Kế hoạch đã được thiết lập thành công dưới dạng cây quyết định.")

    print("\n--- MÔ PHỎNG QUÁ TRÌNH CHẠY THỰC TẾ ---")
    print("Ký hiệu: M là vị trí THỰC, [...] là các vị trí robot NGHĨ mình có thể ở đó.")

    def print_correct_room(room_state, rx, ry, beliefs):
        for i in range(m):
            for j in range(n):
                is_real = (i == rx and j == ry)
                is_believed = ((i, j) in beliefs)
                val = room_state[i][j]
                if is_real and is_believed:
                    print(f"[{val if val != 0 else 'M'}]", end=" ")
                elif is_real:
                    print(f" M ", end=" ")
                elif is_believed:
                    print(f"[{val}]", end=" ")
                else:
                    print(f" {val} ", end=" ")
            print()

    curr_room_state = [list(row) for row in room]
    curr_real_x, curr_real_y = real_x, real_y
    curr_beliefs = frozenset((i, j) for i in range(m) for j in range(n))
    curr_room_tuple = tuple(tuple(row) for row in curr_room_state)

    current_plan_node = plan
    step = 1

    print("Trạng thái bắt đầu:")
    print_correct_room(curr_room_state, curr_real_x, curr_real_y, curr_beliefs)
    print(f"Robot đoán mình có thể ở {len(curr_beliefs)} ô.")
    print("-" * 45)

    while current_plan_node != "GOAL" and current_plan_node is not None:
        action, branches = current_plan_node
        print(f"Bước {step}: Hành động được chọn theo kế hoạch -> [{action}]")

        if action == "UP":        curr_real_x = max(0, curr_real_x - 1)
        elif action == "DOWN":    curr_real_x = min(m - 1, curr_real_x + 1)
        elif action == "LEFT":    curr_real_y = max(0, curr_real_y - 1)
        elif action == "RIGHT":   curr_real_y = min(n - 1, curr_real_y + 1)

        curr_room_state[curr_real_x][curr_real_y] = 0

        real_percept = get_percept(curr_real_x, curr_real_y)
        print(f"Cảm biến trả về (U, D, L, R): {real_percept}")
        pred_beliefs = predict_belief(curr_beliefs, action)
        curr_room_tuple, curr_beliefs = update_room_and_beliefs(curr_room_tuple, pred_beliefs, real_percept)

        print_correct_room(curr_room_state, curr_real_x, curr_real_y, curr_beliefs)
        print(f"Robot thu hẹp phạm vi nghi ngờ xuống còn: {len(curr_beliefs)} ô.")
        print("-" * 45)

        if real_percept in branches:
            current_plan_node = branches[real_percept]
        else:
            print("LỖI: Cảm biến nhận dạng một trường hợp nằm ngoài kế hoạch dự phòng!")
            break
        step += 1

    if current_plan_node == "GOAL":
        print("HOÀN THÀNH: Phòng đã sạch hoàn toàn!")


Vi tri thuc te ban dau của Robot: (0, 1)

--- Tìm kiếm với quan sát một phần ---
Thành công! Đã duyệt 22 trạng thái niềm tin.
Kế hoạch đã được thiết lập thành công dưới dạng cây quyết định.

--- MÔ PHỎNG QUÁ TRÌNH CHẠY THỰC TẾ ---
Ký hiệu: M là vị trí THỰC, [...] là các vị trí robot NGHĨ mình có thể ở đó.
Trạng thái bắt đầu:
[0] [M] [0] 
[1] [0] [1] 
[0] [0] [0] 
Robot đoán mình có thể ở 9 ô.
---------------------------------------------
Bước 1: Hành động được chọn theo kế hoạch -> [UP]
Cảm biến trả về (U, D, L, R): ('W', 'O', 'O', 'O')
 0  [M]  0  
 1   0   1  
 0   0   0  
Robot thu hẹp phạm vi nghi ngờ xuống còn: 1 ô.
---------------------------------------------
Bước 2: Hành động được chọn theo kế hoạch -> [DOWN]
Cảm biến trả về (U, D, L, R): ('O', 'O', 'O', 'O')
 0   0   0  
 1  [M]  1  
 0   0   0  
Robot thu hẹp phạm vi nghi ngờ xuống còn: 1 ô.
---------------------------------------------
Bước 3: Hành động được chọn theo kế hoạch -> [DOWN]
Cảm biến trả về (U, D, L, R): ('O', '